# Task 1: Represent Reviews Using Word2Vec

In [4]:
import os
import urllib.request
import pandas as pd
import numpy as np
import torch
import gensim
import re
from sklearn.model_selection import train_test_split

# 1. Automatically download the Yelp dataset file if missing
url = "https://raw.githubusercontent.com/Microsoft/ML-Server-Python-Samples/master/microsoftml/202/data/sentiment_analysis/yelp_labelled.txt"
filename = "07-yelp-dataset.txt"

if not os.path.exists(filename):
    print("Downloading 07-yelp-dataset.txt...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete!")

# 2. Load the Yelp Dataset
df = pd.read_csv(filename, sep="\t", names=["review", "label"])
print("Number of reviews:", len(df))

# 3. Split the Data (80% train, 20% test)
X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

# ==========================================
# Task 1 Code: Word2Vec Representation & Tensors
# ==========================================

# Tokenize training and testing reviews
train_tokens = [re.findall(r'\b\w+\b', str(text).lower()) for text in X_train]
test_tokens = [re.findall(r'\b\w+\b', str(text).lower()) for text in X_test]

# Train Word2Vec model using the training reviews
w2v_model = gensim.models.Word2Vec(
    sentences=train_tokens, 
    vector_size=200, 
    window=5, 
    min_count=1, 
    sg=1
)

# Function to average word vectors into a single vector per review
def review_to_vector(tokens, model, vector_size=200):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# Convert all train and test reviews to vectors
X_train_w2v = np.array([review_to_vector(tokens, w2v_model, 200) for tokens in train_tokens])
X_test_w2v = np.array([review_to_vector(tokens, w2v_model, 200) for tokens in test_tokens])

# Convert vectors into PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print("\nTask 1 Complete!")
print("Training Word2Vec tensor shape:", X_train_w2v_tensor.shape)
print("Testing Word2Vec tensor shape:", X_test_w2v_tensor.shape)

Download complete!
Number of reviews: 1000
Training reviews: 800
Testing reviews: 200

Task 1 Complete!
Training Word2Vec tensor shape: torch.Size([800, 200])
Testing Word2Vec tensor shape: torch.Size([200, 200])


# Task 2: Build a Different Neural Network

In [5]:
import torch.nn as nn
from sklearn.metrics import accuracy_score

input_size_w2v = X_train_w2v_tensor.shape[1]

# Build the neural network with three hidden layers
model_w2v = nn.Sequential(
    nn.Linear(input_size_w2v, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_w2v.parameters(), lr=0.001)

epochs = 15

# Train the model
for epoch in range(epochs):
    model_w2v.train()
    
    # 1. Forward pass
    outputs = model_w2v(X_train_w2v_tensor)
    
    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)
    
    # 3. Backpropagation
    optimizer.zero_grad()
    loss.backward()
    
    # 4. Update weights
    optimizer.step()
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

# Evaluate and calculate test accuracy
model_w2v.eval()
with torch.no_grad():
    test_outputs = model_w2v(X_test_w2v_tensor)
    test_probabilities = torch.sigmoid(test_outputs)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)
print(f"\nTest Accuracy with Word2Vec: {accuracy:.3f}")

Epoch 1/15 - Loss: 0.6933
Epoch 5/15 - Loss: 0.6931
Epoch 10/15 - Loss: 0.6931
Epoch 15/15 - Loss: 0.6931

Test Accuracy with Word2Vec: 0.480
